#### Tokenization – From Words to Subwords

In this lesson, we explore how text is split into tokens, the building blocks for Large Language Models (LLMs). We will compare classical word tokenization with modern subword methods (BPE, WordPiece) and discuss why tokenization affects model cost and context size.


**Tools/Libraries:** spaCy, Hugging Face Tokenizers

#### Import Required Libraries

We will use spaCy for classical word tokenization and Hugging Face's `tokenizers` library for subword tokenization (BPE, WordPiece).

If not already installed, the following cells will install the required packages.

In [1]:
# Install required libraries if needed
!pip install -q spacy tokenizers

In [2]:
import spacy
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors


#### Classical Word Tokenization with spaCy

Let's use spaCy to perform word-level tokenization on a sample sentence. This is the traditional approach used in many NLP pipelines.


In [4]:
# Download spaCy model if not present
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 3.2 MB/s eta 0:00:04
     ---- ----------------------------------- 1.6/12.8 MB 3.0 MB/s eta 0:00:04
     ------- -------------------------------- 2.4/12.8 MB 3.3 MB/s eta 0:00:04
     --------- ------------------------------ 3.1/12.8 MB 3.4 MB/s eta 0:00:03
     ------------ --------------------------- 3.9/12.8 MB 3.5 MB/s eta 0:00:03
     -------------- ------------------------- 4.7/12.8 MB 3.6 MB/s eta 0:00:03
     ------------------ --------------------- 5.8/12.8 MB 3.7 MB/s eta 0:00:02
     -------------------- ------------------- 6.6/12.8 MB 3.8 MB/s eta 0:00:02
     ----------------------- ---------------- 7.6/12.8 MB 3.9 MB/s eta 0:00:02
     -------------------------- ------------- 8.4/12.8 MB 3.9 MB/s eta 0:00:02
     ---------------------------- ----------- 9.2/12.8 MB 3.9 MB/s

In [5]:
import spacy
nlp = spacy.load('en_core_web_sm')

sample_text = "Tokenization is essential for LLMs. It splits text into smaller pieces."
doc = nlp(sample_text)
print("spaCy Word Tokens:", [token.text for token in doc])

spaCy Word Tokens: ['Tokenization', 'is', 'essential', 'for', 'LLMs', '.', 'It', 'splits', 'text', 'into', 'smaller', 'pieces', '.']


#### Subword Tokenization with Hugging Face Tokenizers (BPE)

Now let's use the Hugging Face `tokenizers` library to train and apply a Byte-Pair Encoding (BPE) tokenizer. BPE splits words into subword units, allowing the model to handle rare and unknown words more effectively.


In [10]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# Prepare training data
bpe_corpus = [sample_text]
# Initialize a BPE tokenizer
bpe_tokenizer = Tokenizer(models.BPE())
bpe_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=50, min_frequency=1, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
bpe_tokenizer.train_from_iterator(bpe_corpus, trainer)

# Encode the sample text
bpe_output = bpe_tokenizer.encode(sample_text)
print("BPE Subword Tokens:", bpe_output.tokens)

BPE Subword Tokens: ['To', 'ken', 'iz', 'ati', 'on', 'is', 'es', 's', 'en', 'ti', 'al', 'fo', 'r', 'LL', 'Ms', '.', 'It', 's', 'p', 'lit', 's', 't', 'ex', 't', 'in', 't', 'o', 's', 'mal', 'ler', 'p', 'ieces', '.']


#### Subword Tokenization with Hugging Face Tokenizers (WordPiece)

WordPiece is another popular subword tokenization method, used in models like BERT. Let's train and apply a WordPiece tokenizer on the same sample text.


In [14]:
from tokenizers import models, trainers

# Initialize a WordPiece tokenizer
wp_tokenizer = Tokenizer(models.WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
wp_trainer = trainers.WordPieceTrainer(vocab_size=10, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
wp_tokenizer.train_from_iterator([sample_text], wp_trainer)

# Encode the sample text
wp_output = wp_tokenizer.encode(sample_text)
print("WordPiece Subword Tokens:", wp_output.tokens)

WordPiece Subword Tokens: ['T', '##o', '##k', '##e', '##n', '##i', '##z', '##a', '##t', '##i', '##o', '##n', 'i', '##s', 'e', '##s', '##s', '##e', '##n', '##t', '##i', '##a', '##l', 'f', '##o', '##r', 'L', '##L', '##M', '##s', '.', 'I', '##t', 's', '##p', '##l', '##i', '##t', '##s', 't', '##e', '##x', '##t', 'i', '##n', '##t', '##o', 's', '##m', '##a', '##l', '##l', '##e', '##r', 'p', '##i', '##e', '##c', '##e', '##s', '.']


#### Comparing Tokenization Outputs

Let's compare the outputs of word, BPE, and WordPiece tokenization on the same text. Notice the differences in the number and type of tokens produced by each method.


In [7]:
# Compare all tokenization outputs
print("spaCy Word Tokens:", [token.text for token in doc])
print("BPE Subword Tokens:", bpe_output.tokens)
print("WordPiece Subword Tokens:", wp_output.tokens)

print(f"\nToken counts: spaCy={len([token.text for token in doc])}, BPE={len(bpe_output.tokens)}, WordPiece={len(wp_output.tokens)}")

spaCy Word Tokens: ['Tokenization', 'is', 'essential', 'for', 'LLMs', '.', 'It', 'splits', 'text', 'into', 'smaller', 'pieces', '.']
BPE Subword Tokens: ['To', 'ken', 'iz', 'ati', 'on', 'is', 'es', 's', 'en', 'ti', 'al', 'fo', 'r', 'LL', 'Ms', '.', 'It', 's', 'p', 'lit', 's', 't', 'ex', 't', 'in', 't', 'o', 's', 'mal', 'ler', 'p', 'ieces', '.']
WordPiece Subword Tokens: ['To', '##k', '##e', '##n', '##i', '##z', '##a', '##t', '##i', '##o', '##n', 'i', '##s', 'es', '##s', '##e', '##nt', '##i', '##al', 'fo', '##r', 'LL', '##M', '##s', '.', 'It', 's', '##p', '##l', '##i', '##t', '##s', 't', '##e', '##x', '##t', 'i', '##nt', '##o', 's', '##m', '##al', '##l', '##e', '##r', 'p', '##i', '##e', '##c', '##e', '##s', '.']

Token counts: spaCy=13, BPE=33, WordPiece=52


#### Analyzing Tokenization Impact on Model Cost and Context Size

Tokenization affects:
- **Model Cost:** More tokens = higher computation and cost for LLMs (especially in APIs that charge per token)
- **Context Size:** LLMs have a maximum number of tokens they can process at once (context window)

**Example Calculation:**
- If a model has a 2048-token context window, and your text is split into 500 tokens by BPE but only 350 by word tokenization, you can fit more content with word tokenization. However, subword tokenization is more robust for rare words and languages.

**Illustration:**

| Method      | Token Count | Pros                        | Cons                        |
|-------------|-------------|-----------------------------|-----------------------------|
| Word        |    X        | Simple, fast                | Fails on rare/unknown words |
| BPE         |    Y        | Handles rare words, compact | Sometimes splits too much   |
| WordPiece   |    Z        | Used in BERT, robust        | Similar to BPE              |

Try changing the sample text and rerun the above cells to see how token counts change!